<a href="https://colab.research.google.com/github/pnperl/Travel_Router/blob/main/Travel_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# LIVE INDIAN RAILWAY HOPPING ENGINE
# ============================================================

!pip -q install pandas requests networkx tabulate

import pandas as pd
import requests
import networkx as nx
from tabulate import tabulate
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================

SOURCE = "JAM"
DESTINATION = "MMCT"

GROUP_SIZE = 10

DATE = "2026-05-12"

API_KEY = "YOUR_API_KEY"

# ============================================================
# API CALL
# ============================================================

def get_trains_between(src, dst):

    url = (
        f"https://indianrailapi.com/api/v2/"
        f"TrainBetweenStation/apikey/{API_KEY}/"
        f"From/{src}/To/{dst}/Date/{DATE}/"
    )

    r = requests.get(url)

    if r.status_code != 200:
        return []

    data = r.json()

    trains = []

    for t in data.get("Trains", []):

        trains.append({

            "train_no": t.get("TrainNo"),
            "train_name": t.get("TrainName"),

            "from": src,
            "to": dst,

            "dep": t.get("DepartureTime"),
            "arr": t.get("ArrivalTime"),

            "classes": t.get("AvailableClasses", [])
        })

    return trains

# ============================================================
# LIVE AVAILABILITY
# ============================================================

def get_availability(train_no, src, dst, quota="GN"):

    url = (
        f"https://indianrailapi.com/api/v2/"
        f"SeatAvailability/apikey/{API_KEY}/"
        f"TrainNumber/{train_no}/"
        f"From/{src}/To/{dst}/"
        f"Date/{DATE}/Quota/{quota}"
    )

    r = requests.get(url)

    if r.status_code != 200:
        return []

    data = r.json()

    results = []

    for row in data.get("Availability", []):

        results.append({

            "class": row["Class"],
            "status": row["Status"]
        })

    return results

# ============================================================
# STATION HUBS
# ============================================================

HUBS = [
    "RJT",
    "ADI",
    "BRC",
    "ST"
]

# ============================================================
# FETCH ROUTES
# ============================================================

all_routes = []

# DIRECT
direct = get_trains_between(SOURCE, DESTINATION)

for t in direct:

    avail = get_availability(
        t["train_no"],
        SOURCE,
        DESTINATION
    )

    all_routes.append({

        "type": "DIRECT",

        "route":
            f"{SOURCE} -> {DESTINATION}",

        "train":
            t["train_name"],

        "availability":
            avail
    })

# HOPPING
for hub in HUBS:

    leg1 = get_trains_between(SOURCE, hub)

    leg2 = get_trains_between(hub, DESTINATION)

    for a in leg1:
        for b in leg2:

            all_routes.append({

                "type": "HOPPING",

                "route":
                    f"{SOURCE} -> {hub} -> {DESTINATION}",

                "leg1":
                    a["train_name"],

                "leg2":
                    b["train_name"]
            })

# ============================================================
# DISPLAY
# ============================================================

df = pd.DataFrame(all_routes)

print(tabulate(df, headers="keys", tablefmt="pretty"))